In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm
from utils.knowledge_check import knowledge_check_truthfulqa, knowledge_check_mmlu
from utils.generation import generate_response, NEUTRAL_SYSTEM
from utils.activation import extract_activations
from utils.probe import probe_all_layers, train_linear_probe
import warnings
warnings.filterwarnings("ignore")   # suppress the greedy-decoding flag warnings

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DATA_DIR = Path("data/dataset")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

Device: cuda
GPU: NVIDIA GeForce RTX 4090
VRAM: 25.4 GB


In [2]:
deception_df = pd.read_csv(DATA_DIR / "deception_dataset.csv")
print(f"deception_dataset: {deception_df.shape}")
print(deception_df["label"].value_counts())

deception_dataset: (400, 6)
label
honest       200
deceptive    200
Name: count, dtype: int64


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

N_LAYERS = model.config.num_hidden_layers
HIDDEN_DIM = model.config.hidden_size
print(f"Loaded: {N_LAYERS} layers, hidden_dim={HIDDEN_DIM}")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [12]:
tqa_mc = load_dataset("truthful_qa", "multiple_choice", split="validation")
TRUTHFULQA_PATH = DATA_DIR / "truthfulQA_test_results.csv"
CHECKPOINT_EVERY = 50

if TRUTHFULQA_PATH.exists():
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    if len(kc_df) == len(tqa_mc):
        print(f"Already complete ({len(kc_df)} rows). Loading from {TRUTHFULQA_PATH}")
    else:
        done_questions = set(kc_df["question"].tolist())
        remaining = [item for item in tqa_mc if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="TruthfulQA knowledge check")):
            records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=False, index=False)
        kc_df = pd.read_csv(TRUTHFULQA_PATH)
        print(f"Done. Total: {len(kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(tqa_mc, desc="TruthfulQA knowledge check")):
        records.append(knowledge_check_truthfulqa(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(TRUTHFULQA_PATH, mode="a", header=not TRUTHFULQA_PATH.exists(), index=False)
    kc_df = pd.read_csv(TRUTHFULQA_PATH)
    print(f"Done. Total: {len(kc_df)}  |  Passed: {kc_df['passed'].sum()}  |  Failed: {(~kc_df['passed']).sum()}")

passed_df = kc_df[kc_df["passed"] == True].reset_index(drop=True)
failed_df = kc_df[kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(passed_df)}  |  Failed: {len(failed_df)}")

Already complete (817 rows). Loading from data/dataset/truthfulQA_test_results.csv
Passed: 398  |  Failed: 419


In [13]:
mmlu = load_dataset("cais/mmlu", "all", split="test")
MMLU_PATH = DATA_DIR / "mmlu_test_results.csv"
CHECKPOINT_EVERY = 50

if MMLU_PATH.exists():
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    if len(mmlu_kc_df) == len(mmlu):
        print(f"Already complete ({len(mmlu_kc_df)} rows). Loading from {MMLU_PATH}")
    else:
        done_questions = set(mmlu_kc_df["question"].tolist())
        remaining = [item for item in mmlu if item["question"] not in done_questions]
        print(f"Resuming from checkpoint: {len(mmlu_kc_df)} done, {len(remaining)} remaining")
        records = []
        for i, item in enumerate(tqdm(remaining, desc="MMLU knowledge check")):
            records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
            if (i + 1) % CHECKPOINT_EVERY == 0:
                pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
                records = []
        if records:
            pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=False, index=False)
        mmlu_kc_df = pd.read_csv(MMLU_PATH)
        print(f"Done. Total: {len(mmlu_kc_df)}")
else:
    records = []
    for i, item in enumerate(tqdm(mmlu, desc="MMLU knowledge check")):
        records.append(knowledge_check_mmlu(item, model, tokenizer, DEVICE))
        if (i + 1) % CHECKPOINT_EVERY == 0:
            df_chunk = pd.DataFrame(records)
            df_chunk.to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
            records = []
    if records:
        pd.DataFrame(records).to_csv(MMLU_PATH, mode="a", header=not MMLU_PATH.exists(), index=False)
    mmlu_kc_df = pd.read_csv(MMLU_PATH)
    print(f"Done. Total: {len(mmlu_kc_df)}  |  Passed: {mmlu_kc_df['passed'].sum()}  |  Failed: {(~mmlu_kc_df['passed']).sum()}")

mmlu_passed_df = mmlu_kc_df[mmlu_kc_df["passed"] == True].reset_index(drop=True)
mmlu_failed_df = mmlu_kc_df[mmlu_kc_df["passed"] == False].reset_index(drop=True)
print(f"Passed: {len(mmlu_passed_df)}  |  Failed: {len(mmlu_failed_df)}")

README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

all/dev-00000-of-00001.parquet:   0%|          | 0.00/76.5k [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

MMLU knowledge check:   0%|          | 0/14042 [00:00<?, ?it/s]

KeyboardInterrupt: 